In [ ]:
# Colab: clone + pip install. Private repo: Colab secret GITHUB_TOKEN (classic PAT, repo) + Notebook access on.

import os
import sys
import shutil
import subprocess

IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
REPO_DIR = os.environ.get("REPO_DIR", "/content/NLP-coursework")
REPO_SLUG = os.environ.get("GITHUB_REPO", "TheFinix13/NLP-coursework").strip().strip("/")
REPO_BRANCH = os.environ.get("REPO_BRANCH", "fiyin/model-pipeline").strip()
REPO_URL = os.environ.get("REPO_URL", "").strip()


def _token():
    t = os.environ.get("GITHUB_TOKEN", "").strip()
    if t:
        return t
    if not IN_COLAB:
        return ""
    try:
        from google.colab import userdata
        v = userdata.get("GITHUB_TOKEN")
        return "" if v is None else str(v).strip()
    except Exception:
        return ""


def _clone_url():
    if REPO_URL:
        return REPO_URL
    tok = _token()
    host = "github.com/" + REPO_SLUG + ".git"
    if tok:
        return "https://oauth2:" + tok + "@" + host
    return "https://" + host


def _is_repo_root(path):
    return bool(
        path
        and os.path.isdir(os.path.join(path, "src"))
        and os.path.isfile(os.path.join(path, "requirements.txt"))
    )


def _find_repo_under_content():
    base = "/content"
    if not os.path.isdir(base):
        return None
    for name in sorted(os.listdir(base)):
        p = os.path.join(base, name)
        if os.path.isdir(p) and _is_repo_root(p):
            return p
    return None


def _ensure_repo_colab():
    if _is_repo_root(REPO_DIR):
        return REPO_DIR

    zip_path = os.environ.get("REPO_ZIP", "/content/NLP-coursework.zip")
    if os.path.isfile(zip_path):
        subprocess.run(["unzip", "-q", "-o", zip_path, "-d", "/content"], check=False)

    found = _find_repo_under_content()
    if found:
        return found

    if os.path.isdir(REPO_DIR) and not _is_repo_root(REPO_DIR):
        shutil.rmtree(REPO_DIR, ignore_errors=True)

    CLONE_URL = _clone_url()
    cmd = ["git", "clone", "--depth", "1"]
    if REPO_BRANCH:
        cmd += ["--branch", REPO_BRANCH]
    cmd += [CLONE_URL, REPO_DIR]

    r = subprocess.run(cmd, capture_output=True, text=True)
    err = (r.stderr or r.stdout or "").strip()

    if r.returncode != 0 or not _is_repo_root(REPO_DIR):
        hint = ""
        if (
            "could not read Username" in err
            or "Authentication failed" in err
            or "403" in err
            or "Repository not found" in err
        ):
            hint = (
                "\nIf the repo is private: Colab → Secrets → GITHUB_TOKEN (repo scope) → enable Notebook access; "
                "Runtime → Restart, then re-run. Or upload NLP-coursework.zip to /content/.\n"
            )
        raise RuntimeError("Clone failed (need src/ + requirements.txt)." + hint + "\ngit said:\n" + err)
    return REPO_DIR


def _ensure_repo_local():
    cur = os.path.abspath(os.getcwd())
    for _ in range(10):
        if _is_repo_root(cur):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError("Run from repo root (contains src/ and requirements.txt).")


if IN_COLAB:
    root = _ensure_repo_colab()
    os.chdir(root)
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install", "-r", "requirements.txt"],
        check=True,
    )
else:
    root = _ensure_repo_local()
    os.chdir(root)

os.environ["NLP_REPO_ROOT"] = os.path.abspath(root)
if root not in sys.path:
    sys.path.insert(0, root)
print("✅ Setup complete —", os.environ["NLP_REPO_ROOT"])



## Task 2.3 — LoRA adapters (Sarcasm)\n
\n
This notebook trains LoRA adapters per variety (en-UK, en-AU, en-IN) on the **Sarcasm** task using a frozen base model in the **1–3B** range.\n
\n
For a fast local check, you can temporarily switch `MODEL_KEY` to a smaller model (e.g., `opt-125m`). For the coursework results, use a 1–3B base (e.g., `qwen2.5-1.5b`).\n

In [ ]:
import os\nimport sys\nimport numpy as np\nimport torch\nfrom datasets import load_dataset\nfrom sklearn.metrics import classification_report, confusion_matrix, f1_score\n\n# Make repo importable (Colab + local)\nPROJECT_ROOT = os.path.abspath(os.getcwd())\nif PROJECT_ROOT not in sys.path:\n    sys.path.insert(0, PROJECT_ROOT)\n\nfrom src.besstie_data_loader import get_variety_split\nfrom src.training_utils import class_weights, new_weighted_class\nfrom models.lora.lora_adapters import (\n    LoRAConfig,\n    load_model,\n    apply_lora,\n    tokenize_dataset,\n    training_args,\n)\n

In [ ]:
# Config\n
TASK = "Sarcasm"\n
VARIETIES = ["en-UK", "en-AU", "en-IN"]\n
\n
# For final coursework runs on GPU, use a 1–3B model like qwen2.5-1.5b\n
MODEL_KEY = os.environ.get("MODEL_KEY", "qwen2.5-1.5b")\n
\n
SEEDS = [42, 123]\n
MAX_LENGTH = 128\n
\n
# Adjust for speed\n
EPOCHS = int(os.environ.get("EPOCHS", "1"))\n
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "4"))\n
LR = float(os.environ.get("LR", "2e-4"))\n
\n
# Optional: reduce training size to speed up on Colab\n
LIMIT_TRAIN = int(os.environ.get("LIMIT_TRAIN", "0"))  # 0 = no limit\n
LIMIT_VAL = int(os.environ.get("LIMIT_VAL", "0"))      # 0 = no limit\n
LIMIT_TEST = int(os.environ.get("LIMIT_TEST", "0"))    # 0 = no limit\n

In [ ]:
ds = load_dataset("surrey-nlp/BESSTIE-CW-26")\n
print(ds)\n

In [ ]:
def maybe_limit(d, n: int, seed: int):\n
    if not n or n <= 0 or n >= len(d):\n
        return d\n
    return d.shuffle(seed=seed).select(range(n))\n
\n
\n
def train_one(variety: str, seed: int):\n
    np.random.seed(seed)\n
    torch.manual_seed(seed)\n
\n
    train = get_variety_split(ds, variety, "train")\n
    val = get_variety_split(ds, variety, "validation")\n
\n
    train = maybe_limit(train, LIMIT_TRAIN, seed)\n
    val = maybe_limit(val, LIMIT_VAL, seed)\n
\n
    base_model, tokenizer = load_model(MODEL_KEY, num_labels=2)\n
    model = apply_lora(base_model, LoRAConfig())\n
\n
    train_tok = tokenize_dataset(train, tokenizer, label_col=TASK, max_length=MAX_LENGTH)\n
    val_tok = tokenize_dataset(val, tokenizer, label_col=TASK, max_length=MAX_LENGTH)\n
\n
    weights = class_weights(train, TASK)\n
    WeightedTrainer = new_weighted_class(weights)\n
    args = training_args(\n
        output_dir=f"./tmp/lora_{variety}_seed{seed}",\n
        variety=variety,\n
        seed=seed,\n
        epochs=EPOCHS,\n
        batch_size=BATCH_SIZE,\n
        lr=LR,\n
    )\n
    args.report_to = "none"\n
    args.save_total_limit = 1\n
\n
    trainer = WeightedTrainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok)\n
    trainer.train()\n
    return trainer, tokenizer\n

In [ ]:
def evaluate(trainer, tokenizer, test_variety: str, seed: int):\n
    test = get_variety_split(ds, test_variety, "test")\n
    test = maybe_limit(test, LIMIT_TEST, seed)\n
    test_tok = tokenize_dataset(test, tokenizer, label_col=TASK, max_length=MAX_LENGTH)\n
\n
    out = trainer.predict(test_tok)\n
    y_pred = np.argmax(out.predictions, axis=1)\n
    y_true = test_tok["labels"].cpu().numpy()\n
\n
    macro_f1 = f1_score(y_true, y_pred, average="macro")\n
    cm = confusion_matrix(y_true, y_pred)\n
    report = classification_report(y_true, y_pred, target_names=["Not Sarcastic", "Sarcastic"])\n
    return {"macro_f1": float(macro_f1), "confusion_matrix": cm.tolist(), "report": report}\n

In [ ]:
# Train adapters per variety + evaluate on each test set\n
results = {}\n
for variety in VARIETIES:\n
    results[variety] = {}\n
    for seed in SEEDS:\n
        trainer, tok = train_one(variety, seed)\n
        seed_res = {}\n
        for test_variety in VARIETIES:\n
            seed_res[test_variety] = evaluate(trainer, tok, test_variety, seed)\n
        results[variety][f"seed_{seed}"] = seed_res\n
\n
results\n